# Trabajo de Fin de Máster — Monitor de Precios Energéticos

En el siguiente notebook se muestra el pipeline completo de extracción, limpieza y almacenamiento de datos energéticos desde ESIOS y OMIE: precios PVPC, mix de generación eléctrica y precios del mercado diario, junto con el cálculo de indicadores derivados como la media reciente del precio y las horas más económicas del día, y un sistema de monitorización que registra cada ejecución del proceso para poder detectar incidencias.

El precio de la electricidad en España varía hora a hora, lo que supone una oportunidad para el consumidor que puede adaptar sus hábitos de consumo a las franjas más económicas del día. Sin embargo, esta información se encuentra dispersa en distintas fuentes, cada una con su propio formato de acceso, lo que dificulta su consulta y seguimiento de forma continuada. A continuación se detalla cada uno de los pasos del proceso.


## 1. Instalación de librerías

In [20]:
!pip install \
    gspread \
    gspread-dataframe \
    pandas \
    requests \
    --quiet

print("Librerías instaladas correctamente")


50fa6519dfe607072baa4e1f89982dd245390e3d259a1f39a1457666217a14f8Librerías instaladas correctamente


## 2. Configuración: token de ESIOS y conexión a Google Sheets

Al ejecutar esta celda se pedirá el token de ESIOS (no se muestra en pantalla ni se guarda en el notebook ya que es un token que proporciona la web de ESIOS para uso personal) y permiso para acceder a Google Drive/Sheets ya que los datos aparecerán automaticamente, una vez ejecutado el notebook, en la Sheet de Google llamada "TFM_Datos_Energía"


In [21]:
import getpass

# Token de ESIOS
ESIOS_TOKEN = getpass.getpass("Introduce tu token de ESIOS: ")

# Nombre de la Google Sheet
SHEET_NAME = "TFM_Datos_Energia"

# Autenticación con Google Sheets
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Abrimos la Sheet y preparamos las pestañas que vamos a usar
sh = gc.open(SHEET_NAME)

def obtener_o_crear_hoja(spreadsheet, nombre, filas=2000, columnas=10):
    try:
        return spreadsheet.worksheet(nombre)
    except gspread.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=nombre, rows=filas, cols=columnas)

hoja_precios = obtener_o_crear_hoja(sh, "precios")
hoja_mix = obtener_o_crear_hoja(sh, "mix")
hoja_omie = obtener_o_crear_hoja(sh, "omie")
hoja_analisis = obtener_o_crear_hoja(sh, "analisis")
hoja_monitor = obtener_o_crear_hoja(sh, "monitorizacion")

print(f"Conectado a la Google Sheet '{SHEET_NAME}")
print(f"Pestañas: '{hoja_precios.title}', '{hoja_mix.title}', '{hoja_omie.title}', '{hoja_analisis.title}', '{hoja_monitor.title}'")


Introduce tu token de ESIOS: ··········
Conectado a la Google Sheet 'TFM_Datos_Energia
Pestañas: 'precios', 'mix', 'omie', 'analisis', 'monitorizacion'


## 3. Función genérica de extracción de indicadores ESIOS

Tanto el PVPC como cada tecnología del mix de generación se piden a ESIOS de la misma forma, cambiando solo el número de indicador. En vez de repetir el código, hacemos una única función genérica y la reutilizamos.

**Nota técnica:** ESIOS migró su infraestructura y ahora exige la cabecera `x-api-key` además de la `Authorization` clásica; sin ella la API devuelve un error 403 aunque el token sea correcto. El código de abajo ya incluye ambas.


In [22]:
import requests
from datetime import datetime

GEO_ID_PENINSULA = 8741  # nos interesa solo Península, no Canarias/Baleares
ESIOS_BASE_URL = "https://api.esios.ree.es"


def extraer_indicador_esios(token: str, indicador_id: int, fecha_inicio: datetime, fecha_fin: datetime) -> list:

    url = f"{ESIOS_BASE_URL}/indicators/{indicador_id}"
    headers = {
        "Accept": "application/json; application/vnd.esios-api-v1+json",
        "Content-Type": "application/json",
        "Host": "api.esios.ree.es",
        "Authorization": f'Token token="{token}"',
        "x-api-key": token,
    }
    params = {
        "start_date": fecha_inicio.isoformat(),
        "end_date": fecha_fin.isoformat(),
    }

    respuesta = requests.get(url, headers=headers, params=params, timeout=30)
    respuesta.raise_for_status()

    datos = respuesta.json()
    return datos.get("indicator", {}).get("values", [])


## 4. Extracción y limpieza: PVPC (indicador 1001)

Esta función se encarga de limpiar los datos brutos del PVPC obtenidos de ESIOS: se queda solo con los valores de Península, ajusta las fechas a la zona horaria de Madrid, elimina nulos y duplicados, y añade el precio también en €/kWh.

In [23]:
import pandas as pd

INDICADOR_PVPC = 1001


def limpiar_pvpc(valores_brutos: list) -> pd.DataFrame:
    if not valores_brutos:
        return pd.DataFrame(columns=["fecha_hora", "precio_eur_mwh", "precio_eur_kwh"])

    df = pd.DataFrame(valores_brutos)
    df = df[df["geo_id"] == GEO_ID_PENINSULA].copy()

    df["fecha_hora"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce").dt.tz_convert("Europe/Madrid")
    df["precio_eur_mwh"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["fecha_hora", "precio_eur_mwh"])
    df = df.drop_duplicates(subset=["fecha_hora"])
    df = df.sort_values("fecha_hora").reset_index(drop=True)

    df["precio_eur_kwh"] = (df["precio_eur_mwh"] / 1000).round(4)

    return df[["fecha_hora", "precio_eur_mwh", "precio_eur_kwh"]]


## 5. Extracción y limpieza: Mix de generación

Pedimos, con la misma función genérica, la generación medida (MW) de tres tecnologías: **solar fotovoltaica**, **eólica** y **ciclo combinado** (gas), dos renovables variables y la tecnología fósil que suele marcar el precio cuando el sol y el viento no llegan a cubrir la demanda.

Estos datos llegan de ESIOS cada 5-15 minutos (según la tecnología), así que los agregamos a media horaria para que se puedan comparar directamente con el PVPC y con OMIE.


In [24]:
INDICADORES_MIX = {
    "Solar fotovoltaica": 1295,
    "Eólica": 2038,
    "Ciclo combinado": 2041,
}


def extraer_y_limpiar_mix(token: str, fecha_inicio: datetime, fecha_fin: datetime) -> pd.DataFrame:

    filas = []
    for nombre_tecnologia, indicador_id in INDICADORES_MIX.items():
        valores = extraer_indicador_esios(token, indicador_id, fecha_inicio, fecha_fin)
        geo_ids_vistos = sorted({v.get("geo_id") for v in valores}, key=lambda x: (x is None, x))
        print(f"  {nombre_tecnologia}: {len(valores)} valores brutos (geo_id presentes: {geo_ids_vistos})")
        for v in valores:
            filas.append({
                "fecha_hora": v.get("datetime"),
                "tecnologia": nombre_tecnologia,
                "generacion_mw": v.get("value"),
            })

    df = pd.DataFrame(filas, columns=["fecha_hora", "tecnologia", "generacion_mw"])
    if df.empty:
        return df

    df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], utc=True, errors="coerce").dt.tz_convert("Europe/Madrid")
    df["generacion_mw"] = pd.to_numeric(df["generacion_mw"], errors="coerce")
    df = df.dropna(subset=["fecha_hora", "generacion_mw"])

    # Agregamos a media horaria: redondeamos cada marca de tiempo a su hora
    # y promediamos todas las lecturas de 5-15 min que caen dentro de ella.
    df["fecha_hora"] = df["fecha_hora"].dt.floor("h")
    df = df.groupby(["fecha_hora", "tecnologia"], as_index=False)["generacion_mw"].mean()
    df["generacion_mw"] = df["generacion_mw"].round(1)

    df = df.sort_values(["fecha_hora", "tecnologia"]).reset_index(drop=True)

    return df


## 6. Extracción y limpieza: OMIE (precio mayorista)

OMIE publica un fichero público diario (sin token) con el precio marginal del mercado. Descargamos un fichero por día del rango solicitado; si algún día concreto todavía no está publicado (por ejemplo, mañana antes de que cierre el mercado), simplemente se omite ese día sin romper el resto.

**Nota sobre la resolución temporal:** desde 2025 OMIE publica el mercado diario en periodos de 15 minutos (hasta 96 al día) en vez de por horas, por una normativa de la CNMC de adaptación al mercado europeo cuarto-horario. El código agrega automáticamente esos 4 periodos de cada hora en su precio medio horario, para poder comparar directamente con el PVPC.


In [25]:
import io
from datetime import timedelta

OMIE_URL = "https://www.omie.es/es/file-download"


def descargar_omie_dia(fecha) -> list:
    """
    Descarga y parsea el fichero de precio marginal de OMIE para un único
    día. Devuelve una lista de dicts: fecha, hora, precio_omie_eur_mwh.

    Desde 2025 (adaptación al mercado cuarto-horario europeo, resolución
    CNMC de febrero de 2025), OMIE publica el mercado diario en periodos
    de 15 minutos: hasta 96 periodos en un día normal, hasta 100 el día
    del cambio de hora de octubre. Como el PVPC sigue siendo horario,
    agregamos cada 4 periodos consecutivos en su precio medio horario,
    para poder comparar ambas fuentes directamente.
    """
    nombre_fichero = f"marginalpdbc_{fecha.strftime('%Y%m%d')}.1"
    params = {"parents[0]": "marginalpdbc", "filename": nombre_fichero}

    respuesta = requests.get(OMIE_URL, params=params, timeout=30)
    respuesta.raise_for_status()
    texto = respuesta.content.decode("latin-1")  # OMIE usa esta codificación, no UTF-8

    tabla = pd.read_csv(
        io.StringIO(texto), sep=";", skiprows=1, header=None,
        engine="python", skip_blank_lines=True,
    )

    periodos = []
    for _, linea in tabla.iterrows():
        try:
            periodo = int(linea[3])
            precio = float(linea[4])
        except (ValueError, TypeError, IndexError):
            continue  # línea de cierre del fichero u otra fila no numérica
        periodos.append({"periodo": periodo, "precio": precio})

    if not periodos:
        return []

    df_periodos = pd.DataFrame(periodos)
    # Los periodos 1-4 son la hora 1, el 5-8 la hora 2, etc.
    df_periodos["hora"] = ((df_periodos["periodo"] - 1) // 4) + 1
    df_horario = df_periodos.groupby("hora", as_index=False)["precio"].mean()

    return [
        {"fecha": fecha.date(), "hora": int(fila["hora"]), "precio_omie_eur_mwh": round(fila["precio"], 2)}
        for _, fila in df_horario.iterrows()
    ]


def extraer_y_limpiar_omie(fecha_inicio: datetime, fecha_fin: datetime) -> pd.DataFrame:
    """
    Descarga los precios de OMIE día a día para el rango indicado. Un fallo
    en un día concreto no interrumpe la descarga de los demás.
    """
    todas_las_filas = []
    dia_actual = fecha_inicio.date()
    while dia_actual <= fecha_fin.date():
        try:
            todas_las_filas.extend(descargar_omie_dia(datetime.combine(dia_actual, datetime.min.time())))
        except requests.RequestException as error:
            print(f"  (OMIE {dia_actual}: todavía no disponible o error de red — se omite: {error})")
        dia_actual += timedelta(days=1)

    df = pd.DataFrame(todas_las_filas, columns=["fecha", "hora", "precio_omie_eur_mwh"])
    if df.empty:
        return df

    df = df.dropna(subset=["precio_omie_eur_mwh"])
    df = df.drop_duplicates(subset=["fecha", "hora"])
    df = df.sort_values(["fecha", "hora"]).reset_index(drop=True)
    return df


## 7. Cálculos derivados: media reciente y horas más baratas

Esto se calcula solo sobre el PVPC, que es el precio que de verdad le importa al usuario.


In [26]:
def calcular_resumen(df: pd.DataFrame, n_horas_baratas: int = 5) -> dict:
    """
    A partir de la tabla limpia de PVPC, calcula:
    - La media de precio de los días anteriores a hoy (contexto reciente)
    - Las horas más baratas de hoy y, si ya están publicadas, de mañana
    """
    hoy = pd.Timestamp.now(tz="Europe/Madrid").normalize()
    manana = hoy + pd.Timedelta(days=1)

    df_pasado = df[df["fecha_hora"].dt.normalize() < hoy]
    df_hoy = df[df["fecha_hora"].dt.normalize() == hoy]
    df_manana = df[df["fecha_hora"].dt.normalize() == manana]

    media_reciente_mwh = round(df_pasado["precio_eur_mwh"].mean(), 2) if not df_pasado.empty else None
    media_reciente_kwh = round(media_reciente_mwh / 1000, 4) if media_reciente_mwh is not None else None
    horas_baratas_hoy = df_hoy.nsmallest(n_horas_baratas, "precio_eur_mwh")
    horas_baratas_manana = df_manana.nsmallest(n_horas_baratas, "precio_eur_mwh")

    return {
        "media_reciente_eur_mwh": media_reciente_mwh,
        "media_reciente_eur_kwh": media_reciente_kwh,
        "horas_baratas_hoy": horas_baratas_hoy,
        "horas_baratas_manana": horas_baratas_manana,
        "manana_disponible": not df_manana.empty,
    }


## 8. Guardar en la Google Sheet (sin duplicados)

Misma idea para las tres fuentes: antes de escribir, comprobamos qué ya estaba guardado, para poder re-ejecutar el pipeline sin duplicar filas. Cada función compara **fechas ya interpretadas como fecha** (no como texto), porque Google Sheets reformatea las fechas automáticamente y comparar texto contra texto puede dar duplicados — ya nos pasó una vez con el PVPC.


In [27]:
from gspread_dataframe import get_as_dataframe, set_with_dataframe

COLUMNAS_PRECIOS = ["fecha_hora", "precio_eur_mwh", "precio_eur_kwh"]
COLUMNAS_MIX = ["fecha_hora", "tecnologia", "generacion_mw"]
COLUMNAS_OMIE = ["fecha", "hora", "precio_omie_eur_mwh"]


def guardar_precios_sin_duplicar(hoja, df_nuevo: pd.DataFrame) -> int:
    """Guarda el PVPC evitando duplicados (clave: fecha_hora)."""
    df_para_guardar = df_nuevo[COLUMNAS_PRECIOS].copy()
    df_para_guardar["fecha_hora"] = df_para_guardar["fecha_hora"].dt.strftime("%Y-%m-%d %H:%M:%S")

    df_existente = get_as_dataframe(hoja, evaluate_formulas=True).dropna(how="all")
    if df_existente.empty or not set(COLUMNAS_PRECIOS).issubset(df_existente.columns):
        set_with_dataframe(hoja, df_para_guardar)
        return len(df_para_guardar)

    df_existente = df_existente[COLUMNAS_PRECIOS]
    fechas_existentes = set(pd.to_datetime(df_existente["fecha_hora"], errors="coerce"))
    fechas_nuevas = pd.to_datetime(df_para_guardar["fecha_hora"], errors="coerce")

    df_a_anadir = df_para_guardar[~fechas_nuevas.isin(fechas_existentes)]
    if df_a_anadir.empty:
        return 0

    df_final = pd.concat([df_existente, df_a_anadir], ignore_index=True)
    set_with_dataframe(hoja, df_final)
    return len(df_a_anadir)


def guardar_mix_sin_duplicar(hoja, df_nuevo: pd.DataFrame) -> int:
    """Guarda el mix de generación evitando duplicados (clave: fecha_hora + tecnología)."""
    df_para_guardar = df_nuevo[COLUMNAS_MIX].copy()
    df_para_guardar["fecha_hora"] = df_para_guardar["fecha_hora"].dt.strftime("%Y-%m-%d %H:%M:%S")

    df_existente = get_as_dataframe(hoja, evaluate_formulas=True).dropna(how="all")
    if df_existente.empty or not set(COLUMNAS_MIX).issubset(df_existente.columns):
        set_with_dataframe(hoja, df_para_guardar)
        return len(df_para_guardar)

    df_existente = df_existente[COLUMNAS_MIX]

    def clave(df):
        fecha_hora = pd.to_datetime(df["fecha_hora"], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
        return fecha_hora + "_" + df["tecnologia"].astype(str)

    claves_existentes = set(clave(df_existente))
    df_a_anadir = df_para_guardar[~clave(df_para_guardar).isin(claves_existentes)]
    if df_a_anadir.empty:
        return 0

    df_final = pd.concat([df_existente, df_a_anadir], ignore_index=True)
    set_with_dataframe(hoja, df_final)
    return len(df_a_anadir)


def guardar_omie_sin_duplicar(hoja, df_nuevo: pd.DataFrame) -> int:
    """Guarda los precios de OMIE evitando duplicados (clave: fecha + hora)."""
    df_para_guardar = df_nuevo[COLUMNAS_OMIE].copy()

    df_existente = get_as_dataframe(hoja, evaluate_formulas=True).dropna(how="all")
    if df_existente.empty or not set(COLUMNAS_OMIE).issubset(df_existente.columns):
        set_with_dataframe(hoja, df_para_guardar)
        return len(df_para_guardar)

    df_existente = df_existente[COLUMNAS_OMIE]

    def clave(df):
        fecha = pd.to_datetime(df["fecha"], errors="coerce").dt.strftime("%Y-%m-%d")
        hora = pd.to_numeric(df["hora"], errors="coerce").astype("Int64").astype(str)
        return fecha + "_" + hora

    claves_existentes = set(clave(df_existente))
    df_a_anadir = df_para_guardar[~clave(df_para_guardar).isin(claves_existentes)]
    if df_a_anadir.empty:
        return 0

    df_final = pd.concat([df_existente, df_a_anadir], ignore_index=True)
    set_with_dataframe(hoja, df_final)
    return len(df_a_anadir)


## 9. Control de calidad, formato visual y tabla combinada

Tres mejoras:
- Un **detector de huecos**: compara las horas que deberíamos tener con las que realmente llegaron.
- Un **formato de color** en la Sheet (verde=barato, rojo=caro) para que se lea de un vistazo.
- Una **tabla combinada** de precio + mix, cruzando ambas fuentes solo en las horas donde las dos existen (ya que los datos de generación de MWh por tipo de energía no se publican hasta cierto tiempo después de la hora, es decir, los MWh generados por cada energía a las 9.00h no seran publicados hasta pasadas las 9.00h (normalmente suele tardar un rando de 1-2 horas en publicarse)).


In [28]:
def contar_huecos_horarios(fechas_horas) -> int:
    """
    Dada una serie de fechas_hora ya limpias, comprueba si faltan horas
    entre la primera y la última recibida. Devuelve cuántas faltan (0 si
    no hay huecos). Es una comprobación interna: no asume nada sobre qué
    horas 'deberían' existir en el mundo real, solo sobre el propio rango
    de datos que se ha recibido.
    """
    fechas_horas = pd.Series(fechas_horas).dropna().unique()
    if len(fechas_horas) < 2:
        return 0
    fechas_horas = pd.to_datetime(sorted(fechas_horas))
    esperadas = pd.date_range(start=fechas_horas[0], end=fechas_horas[-1], freq="h")
    return len(esperadas) - len(fechas_horas)


def aplicar_escala_color_precio(hoja) -> None:
    """
    Colorea la columna precio_eur_kwh de la pestaña de precios (verde=barato,
    rojo=caro), según el mínimo y máximo de todos los datos ya guardados.
    Se recalcula entera cada vez que se llama, así que no hace falta
    preocuparse por ir acumulando formatos de ejecuciones anteriores.
    """
    valores = hoja.get_all_values()
    if len(valores) < 2:
        return

    cabecera = valores[0]
    if "precio_eur_kwh" not in cabecera:
        return
    col_idx = cabecera.index("precio_eur_kwh")

    precios = []
    for fila in valores[1:]:
        try:
            precios.append(float(fila[col_idx].replace(",", ".")))
        except (ValueError, IndexError):
            precios.append(None)

    validos = [p for p in precios if p is not None]
    if not validos:
        return
    minimo, maximo = min(validos), max(validos)
    rango = (maximo - minimo) or 1  # evita dividir por 0 si todos los precios son iguales

    requests = []
    for i, precio in enumerate(precios):
        if precio is None:
            continue
        t = (precio - minimo) / rango  # 0 = más barato, 1 = más caro
        color = {
            "red": 0.71 + t * (0.96 - 0.71),
            "green": 0.88 + t * (0.60 - 0.88),
            "blue": 0.71 + t * (0.60 - 0.71),
        }
        requests.append({
            "repeatCell": {
                "range": {
                    "sheetId": hoja.id,
                    "startRowIndex": i + 1,
                    "endRowIndex": i + 2,
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                },
                "cell": {"userEnteredFormat": {"backgroundColor": color}},
                "fields": "userEnteredFormat.backgroundColor",
            }
        })

    if requests:
        hoja.spreadsheet.batch_update({"requests": requests})


def combinar_precio_y_mix(df_pvpc: pd.DataFrame, df_mix: pd.DataFrame) -> pd.DataFrame:
    """
    Cruza el precio (PVPC) con el mix de generación, quedándose solo con
    las horas donde AMBAS fuentes tienen dato — nunca se extiende el mix
    hacia horas futuras que todavía no existen.
    """
    if df_pvpc.empty or df_mix.empty:
        return pd.DataFrame()

    mix_ancho = df_mix.pivot(index="fecha_hora", columns="tecnologia", values="generacion_mw").reset_index()
    combinado = pd.merge(df_pvpc, mix_ancho, on="fecha_hora", how="inner")
    return combinado.sort_values("fecha_hora").reset_index(drop=True)


def guardar_tabla_combinada(hoja, df_combinado: pd.DataFrame) -> None:
    """
    Sobrescribe la pestaña de análisis con la tabla combinada recién
    calculada. No hace falta lógica anti-duplicados: al ser una tabla
    derivada de datos que ya tenemos guardados de forma segura en otras
    pestañas, siempre se puede recalcular entera desde cero.
    """
    hoja.clear()
    if df_combinado.empty:
        return
    df_para_guardar = df_combinado.copy()
    df_para_guardar["fecha_hora"] = df_para_guardar["fecha_hora"].dt.strftime("%Y-%m-%d %H:%M:%S")
    set_with_dataframe(hoja, df_para_guardar)


## 10. Registro de monitorización

Cada fuente registra su propia fila (fecha, si fue bien, cuántas filas nuevas, y de qué fuente se trata), para poder auditar cada una por separado.


In [29]:
from zoneinfo import ZoneInfo


def registrar_ejecucion(hoja_monitor, estado: str, filas_nuevas: int, mensaje: str = "") -> None:

    if hoja_monitor.acell("A1").value != "fecha_ejecucion":
        hoja_monitor.insert_row(["fecha_ejecucion", "estado", "filas_nuevas", "mensaje"], index=1)

    fila = [
        datetime.now(ZoneInfo("Europe/Madrid")).isoformat(timespec="seconds"),
        estado,
        filas_nuevas,
        mensaje,
    ]
    hoja_monitor.append_row(fila)


## 11. Ejecución completa del pipeline

Cada fuente se ejecuta de forma **independiente**: si una falla (por ejemplo, OMIE de mañana todavía no publicado), las otras dos siguen guardándose con normalidad, y el fallo queda igualmente registrado en la monitorización.

`DIAS_HISTORICO` es el parámetro configurable de periodicidad.


In [30]:
DIAS_HISTORICO = 7  # días hacia atrás usados para el histórico de las tres fuentes

fecha_fin = datetime.now() + timedelta(days=1)      # hasta el dia siguiente, por si ya está publicado
fecha_inicio = datetime.now() - timedelta(days=DIAS_HISTORICO)

print(f"Rango de extracción: {fecha_inicio.date()} → {fecha_fin.date()}\n")

# 1) PVPC
try:
    print("PVPC (ESIOS)")
    valores_brutos = extraer_indicador_esios(ESIOS_TOKEN, INDICADOR_PVPC, fecha_inicio, fecha_fin)
    df_pvpc = limpiar_pvpc(valores_brutos)
    filas_nuevas_pvpc = guardar_precios_sin_duplicar(hoja_precios, df_pvpc)
    huecos_pvpc = contar_huecos_horarios(df_pvpc["fecha_hora"])
    print(f"Filas nuevas guardadas: {filas_nuevas_pvpc}")
    if huecos_pvpc:
        print(f"  ⚠️ Se detectan {huecos_pvpc} horas con hueco en el rango recibido")
    aplicar_escala_color_precio(hoja_precios)
    mensaje_pvpc = "PVPC" + (f" — {huecos_pvpc} horas con hueco" if huecos_pvpc else "")
    registrar_ejecucion(hoja_monitor, estado="OK", filas_nuevas=filas_nuevas_pvpc, mensaje=mensaje_pvpc)
except Exception as error:
    mensaje_error = f"{type(error).__name__}: {error}"
    print(f"❌ Error en PVPC: {mensaje_error}")
    registrar_ejecucion(hoja_monitor, estado="ERROR", filas_nuevas=0, mensaje=f"PVPC: {mensaje_error}")
    df_pvpc = pd.DataFrame(columns=["fecha_hora", "precio_eur_mwh", "precio_eur_kwh"])

# 2) Mix de generación
df_mix = pd.DataFrame()
try:
    print("\n Mix de generación (ESIOS)")
    df_mix = extraer_y_limpiar_mix(ESIOS_TOKEN, fecha_inicio, fecha_fin)
    filas_nuevas_mix = guardar_mix_sin_duplicar(hoja_mix, df_mix)
    huecos_mix = contar_huecos_horarios(df_mix["fecha_hora"]) if not df_mix.empty else 0
    print(f"Filas nuevas guardadas: {filas_nuevas_mix}")
    if huecos_mix:
        print(f"  ⚠️ Se detectan {huecos_mix} horas con hueco en el rango recibido")
    mensaje_mix = "Mix generación" + (f" — {huecos_mix} horas con hueco" if huecos_mix else "")
    registrar_ejecucion(hoja_monitor, estado="OK", filas_nuevas=filas_nuevas_mix, mensaje=mensaje_mix)
except Exception as error:
    mensaje_error = f"{type(error).__name__}: {error}"
    print(f"❌ Error en Mix: {mensaje_error}")
    registrar_ejecucion(hoja_monitor, estado="ERROR", filas_nuevas=0, mensaje=f"Mix: {mensaje_error}")

# 3) OMIE
try:
    print("\n OMIE (precio mayorista)")
    df_omie = extraer_y_limpiar_omie(fecha_inicio, fecha_fin)
    filas_nuevas_omie = guardar_omie_sin_duplicar(hoja_omie, df_omie)
    if not df_omie.empty:
        fechas_omie = pd.to_datetime(df_omie["fecha"].astype(str)) + pd.to_timedelta(df_omie["hora"] - 1, unit="h")
        huecos_omie = contar_huecos_horarios(fechas_omie)
    else:
        huecos_omie = 0
    print(f"Filas nuevas guardadas: {filas_nuevas_omie}")
    if huecos_omie:
        print(f"  ⚠️ Se detectan {huecos_omie} horas con hueco en el rango recibido")
    mensaje_omie = "OMIE" + (f" — {huecos_omie} horas con hueco" if huecos_omie else "")
    registrar_ejecucion(hoja_monitor, estado="OK", filas_nuevas=filas_nuevas_omie, mensaje=mensaje_omie)
except Exception as error:
    mensaje_error = f"{type(error).__name__}: {error}"
    print(f"❌ Error en OMIE: {mensaje_error}")
    registrar_ejecucion(hoja_monitor, estado="ERROR", filas_nuevas=0, mensaje=f"OMIE: {mensaje_error}")

# 4) Tabla combinada precio + mix (solo horas solapadas)
try:
    print("\n── Tabla combinada (precio + mix) ──")
    df_combinado = combinar_precio_y_mix(df_pvpc, df_mix)
    guardar_tabla_combinada(hoja_analisis, df_combinado)
    print(f"Filas en la tabla combinada: {len(df_combinado)}")
    registrar_ejecucion(hoja_monitor, estado="OK", filas_nuevas=len(df_combinado), mensaje="Tabla combinada")
except Exception as error:
    mensaje_error = f"{type(error).__name__}: {error}"
    print(f"❌ Error en tabla combinada: {mensaje_error}")
    registrar_ejecucion(hoja_monitor, estado="ERROR", filas_nuevas=0, mensaje=f"Combinada: {mensaje_error}")

# Resumen (solo si el PVPC se descargó bien)
if not df_pvpc.empty:
    resumen = calcular_resumen(df_pvpc)
    print(f"\nMedia de precio de días anteriores: {resumen['media_reciente_eur_mwh']} €/MWh  ({resumen['media_reciente_eur_kwh']} €/kWh)")
    print("\nHoras más baratas de hoy:")
    print(resumen["horas_baratas_hoy"].to_string(index=False))
    if resumen["manana_disponible"]:
        print("\nHoras más baratas de mañana:")
        print(resumen["horas_baratas_manana"].to_string(index=False))
    else:
        print("\n(Los precios de mañana todavía no están publicados)")

print("\n Pipeline finalizado")


Rango de extracción: 2026-08-05 → 2026-08-13

PVPC (ESIOS)
Filas nuevas guardadas: 0

 Mix de generación (ESIOS)
  Solar fotovoltaica: 2039 valores brutos (geo_id presentes: [8741])
  Eólica: 2039 valores brutos (geo_id presentes: [3])
  Ciclo combinado: 2039 valores brutos (geo_id presentes: [3])
Filas nuevas guardadas: 0

 OMIE (precio mayorista)
  (OMIE 2026-08-13: todavía no disponible o error de red — se omite: 404 Client Error: Not Found for url: https://www.omie.es/es/file-download?parents=marginalpdbc&filename=marginalpdbc_20260813.1)
Filas nuevas guardadas: 0

── Tabla combinada (precio + mix) ──
Filas en la tabla combinada: 170

Media de precio de días anteriores: 162.1 €/MWh  (0.1621 €/kWh)

Horas más baratas de hoy:
               fecha_hora  precio_eur_mwh  precio_eur_kwh
2026-08-12 14:00:00+02:00           63.11          0.0631
2026-08-12 15:00:00+02:00           71.05          0.0710
2026-08-12 16:00:00+02:00           99.33          0.0993
2026-08-12 13:00:00+02:00     